# Diabetes Hospital Readmission — End-to-End ML Pipeline

## Objective
Build a reusable preprocessing and machine-learning pipeline to predict **readmission within 30 days** using the UCI diabetes hospital dataset.

In this notebook, I:
- create the target and a simple medication-count feature
- separate patients between training and test data to reduce patient-level leakage
- use `Pipeline` and `ColumnTransformer` so learned preprocessing is fitted only on training data
- evaluate an imbalanced binary classifier with suitable metrics
- save the trained preprocessing and model together

This is a learning project, **not a clinically validated prediction tool**.


## Project decisions

The original `readmitted` column contains `<30`, `>30`, and `NO`. I create:

- `1`: readmitted within 30 days (`<30`)
- `0`: not readmitted within 30 days (`>30` or `NO`)

I remove some ID-coded and administrative fields for simplicity. I create `number_of_medications`, the number of diabetes medication columns containing a known value other than `No`. This is different from the original `num_medications` column, which counts all distinct medications.

I exclude encounters ending in death or hospice because the outcome is not comparable for those cases. Features are from the hospital encounter, so this is a **retrospective near-discharge prediction exercise**, not a prediction made before admission.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

RANDOM_STATE = 42


## 1. Load the raw UCI dataset

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"

# Use /content in Colab, or the current folder on a local computer.
data_dir = Path("/content") if Path("/content").exists() else Path.cwd()
zip_path = data_dir / "diabetes_uci.zip"
extract_path = data_dir / "diabetes_uci"
csv_path = extract_path / "diabetic_data.csv"

if not csv_path.exists():
    if not zip_path.exists():
        print("Downloading the public UCI dataset...")
        urllib.request.urlretrieve(url, zip_path)
    extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Basic inspection

In [ ]:

print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nOriginal target:")
display(df["readmitted"].value_counts(dropna=False))


In [ ]:

# Missing values represented by NaN
print("NaN values:")
display(df.isna().sum().sort_values(ascending=False).head(20))

# Missing values represented by '?'
print("'?' values:")
display((df == "?").sum().sort_values(ascending=False).head(20))


## 3. Clean data and create features

These steps use known per-record values, not statistics learned across records:

1. replace `?` with missing values
2. count diabetes medications with a known value other than `No`
3. define the binary 30-day readmission target
4. exclude death and hospice encounters

Imputation, scaling and encoding are learned later **inside the training pipeline**.


In [ ]:
df = df.replace("?", np.nan)

medication_cols = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide",
    "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone"
]

# Missing medication entries should not be counted as prescribed.
known_medications = df[medication_cols].notna()
prescribed_medications = df[medication_cols].ne("No")
df["number_of_medications"] = (known_medications & prescribed_medications).sum(axis=1)

df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

display(df[["number_of_medications", "num_medications", "readmitted", "readmitted_30"]].head(10))



### Excluding death and hospice encounters

The commonly used discharge disposition codes are:

- `11` = expired
- `13` = hospice / home
- `14` = hospice / medical facility

We exclude these because they do not represent a meaningful opportunity for a 30-day readmission.


In [ ]:

excluded_disposition_ids = [11, 13, 14]

before = len(df)
df = df[~df["discharge_disposition_id"].isin(excluded_disposition_ids)].copy()
after = len(df)

print(f"Rows before exclusion: {before:,}")
print(f"Rows after exclusion:  {after:,}")
print(f"Rows removed:          {before - after:,}")


## 4. Define features (`X`) and target (`y`)

I remove target columns, patient/encounter identifiers, selected administrative variables and individual medication columns after creating the medication-count feature.

**Important:** `patient_nbr` is excluded from model input but kept separately to build **patient-disjoint train/test splits and cross-validation folds**. Otherwise, multiple encounters from the same patient could appear in both sets and make evaluation optimistic.


In [ ]:

drop_cols = [
    "readmitted",
    "readmitted_30",
    "encounter_id",
    "patient_nbr",
    "admission_type_id",
    "admission_source_id",
    "payer_code",
    "num_medications",
    "discharge_disposition_id"
] + medication_cols

X = df.drop(columns=drop_cols, errors="ignore")
y = df["readmitted_30"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget counts:")
display(y.value_counts())

print("\nTarget proportions:")
display(y.value_counts(normalize=True))


## 5. Patient-level train/test split

The original dataset can contain multiple encounters for one patient. A random row split may place the same patient in both training and test sets.

Here I use `StratifiedGroupKFold` to select approximately one-fifth of the patients' encounters for the test set. This:

- **groups** all encounters of each patient together
- **approximately stratifies** the 30-day readmission label (exact proportions are not guaranteed)
- keeps the held-out test data separate until final evaluation


In [ ]:
patient_groups = df["patient_nbr"]

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(X, y, groups=patient_groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_patient_ids = patient_groups.iloc[train_idx]
test_patient_ids = patient_groups.iloc[test_idx]
shared_patients = set(train_patient_ids) & set(test_patient_ids)

print("Training:", X_train.shape, y_train.shape)
print("Test:    ", X_test.shape, y_test.shape)
print("Patients present in both sets:", len(shared_patients))
assert len(shared_patients) == 0, "Patient leakage detected!"

print("\nTraining positive class proportion:", round(y_train.mean(), 4))
print("Test positive class proportion:    ", round(y_test.mean(), 4))



# 6. Identify numerical and categorical features

A column containing numbers is not automatically a numerical measurement.

We have already removed several ID-coded categorical variables such as `admission_type_id`.

The remaining columns are divided based on their pandas data type.


In [ ]:

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print(f"\nNumerical feature count: {len(numeric_features)}")
print(f"Categorical feature count: {len(categorical_features)}")



# 7. Numerical preprocessing pipeline

For numerical columns:

**Missing values → median imputation → standardization**

Median is a robust choice for potentially skewed hospital variables.

The scaler and imputer are fitted only on training data because they live inside the pipeline.


In [ ]:

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])



# 8. Categorical preprocessing pipeline

For categorical columns:

**Missing values → most-frequent imputation → one-hot encoding**

`handle_unknown="ignore"` prevents an unseen test category from causing an error.


In [ ]:

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])


# 9. Combine pipelines with ColumnTransformer

In [ ]:

preprocessing = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

#



# 10. Complete machine-learning pipeline

We use logistic regression as a strong, interpretable baseline.

The target is imbalanced, so we use:

`class_weight="balanced"`

This makes the classifier give greater weight to the minority class without generating synthetic observations.

This is safer and simpler here than applying SMOTE after one-hot encoding.


In [ ]:

model_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

model_pipeline


# 11. Train the complete pipeline

In [ ]:

model_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully.")


## 12. Patient-aware cross-validation on training data

During validation, **all encounters for one patient stay in the same fold**. The test set remains unused. The same preprocessing/model pipeline is fitted from scratch on each training fold.

For imbalanced labels, ROC-AUC and Average Precision are more informative than accuracy alone. Average Precision should also be compared with positive-class prevalence.


In [ ]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = cross_validate(
    model_pipeline,
    X_train,
    y_train,
    groups=train_patient_ids,
    cv=cv,
    scoring={"roc_auc": "roc_auc", "average_precision": "average_precision"}
)

cv_auc = cv_results["test_roc_auc"]
cv_ap = cv_results["test_average_precision"]

print("ROC-AUC scores:", np.round(cv_auc, 4))
print("Mean ROC-AUC:   ", round(cv_auc.mean(), 4))
print("\nAverage Precision scores:", np.round(cv_ap, 4))
print("Mean Average Precision:   ", round(cv_ap.mean(), 4))


## 13. Final evaluation on unseen patients

The final model is fitted only on training patients. We now evaluate once on the held-out patient group using accuracy, precision, recall, F1, ROC-AUC, and Average Precision.

**Important:** Using `class_weight="balanced"` changes the precision/recall trade-off. High accuracy or ROC-AUC alone does not make this suitable for clinical use.


In [ ]:

y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Average Precision": average_precision_score(y_test, y_prob)
}

for name, value in metrics.items():
    print(f"{name:20s}: {value:.4f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        "Not readmitted within 30 days",
        "Readmitted within 30 days"
    ],
    zero_division=0
))


### Compare with a simple baseline

A model that always predicts the majority class can have high accuracy in imbalanced data. I compare it with logistic regression to put the results in context. This baseline is fitted on training patients only.

In [ ]:
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

print("Test positive class prevalence:", round(y_test.mean(), 4))
print("Baseline accuracy:           ", round(accuracy_score(y_test, baseline_pred), 4))
print("Baseline average precision:  ", round(average_precision_score(y_test, baseline_prob), 4))
print("Logistic regression accuracy:", round(metrics["Accuracy"], 4))
print("Logistic regression AP:      ", round(metrics["Average Precision"], 4))


# 14. Confusion matrix

In [ ]:

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No 30-day readmission", "30-day readmission"],
    xticks_rotation=20
)
plt.title("Confusion Matrix")
plt.show()


# 15. ROC curve

In [ ]:

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()


# 16. Precision-Recall curve

In [ ]:

PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title("Precision-Recall Curve")
plt.show()



# 17. Inspect the transformed feature space

The raw data contains mixed numerical and categorical information.

After preprocessing:

- numerical features are imputed and standardized
- categorical features are imputed and one-hot encoded

The resulting matrix is what logistic regression actually receives.


In [ ]:

X_train_transformed = model_pipeline.named_steps["preprocessing"].transform(X_train)
X_test_transformed = model_pipeline.named_steps["preprocessing"].transform(X_test)

print("Original training shape:   ", X_train.shape)
print("Transformed training shape:", X_train_transformed.shape)

print("Original test shape:       ", X_test.shape)
print("Transformed test shape:    ", X_test_transformed.shape)



# 18. Inspect model coefficients

Because we are using logistic regression, we can inspect which transformed features have the largest positive or negative coefficients.

This is useful for understanding the model, although one-hot encoded categories should be interpreted carefully.


In [ ]:

feature_names = model_pipeline.named_steps["preprocessing"].get_feature_names_out()
coefficients = model_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()

print("Largest positive coefficients:")
display(
    coef_df.sort_values("coefficient", ascending=False).head(15)
)

print("Largest negative coefficients:")
display(
    coef_df.sort_values("coefficient", ascending=True).head(15)
)



# 19. Reusable prediction function

The major advantage of the complete pipeline is that a new raw record can go through the same preprocessing automatically.

We do not manually impute, scale, or encode the new record.


In [ ]:

def predict_readmission(patient_data):
    predictions = model_pipeline.predict(patient_data)
    probabilities = model_pipeline.predict_proba(patient_data)[:, 1]

    return pd.DataFrame({
        "predicted_readmission_30": predictions,
        "probability_readmission_30": probabilities
    })

# Example
predict_readmission(X_test.head())



# 20. Save the complete pipeline

We save the **entire pipeline**, not just the classifier.

Therefore the saved object contains:

- numerical imputation
- scaling
- categorical imputation
- one-hot encoding
- logistic regression

This makes the model reusable later.


In [ ]:
import joblib

model_path = data_dir / "diabetes_readmission_pipeline.joblib"
joblib.dump(model_pipeline, model_path)

print(f"Saved to: {model_path}")


## 21. Final pipeline summary

```text
UCI DIABETES DATASET
   ↓
Replace '?' with NaN → feature engineering → define label
   ↓
Exclude death/hospice encounters
   ↓
Keep patient_nbr separately as patient group
   ↓
PATIENT-DISJOINT TRAIN/TEST SPLIT
   ↓
TRAINING PATIENTS
   ├── Numerical features: median imputation → scaling
   ├── Categorical features: most-frequent imputation → one-hot encoding
   └── ColumnTransformer → Logistic Regression (balanced class weights)
   ↓
PATIENT-AWARE CROSS-VALIDATION (training data only)
   ↓
FINAL TEST ON UNSEEN PATIENTS
   ↓
Compare against majority-class baseline → Save complete pipeline
```

### Main lessons

1. Split by **patient**, not encounter, when several rows may belong to one person.
2. Keep the test set separate until evaluation.
3. Put learned preprocessing inside `Pipeline` / `ColumnTransformer`.
4. Evaluate imbalanced models with precision, recall, ROC-AUC, and Average Precision, not accuracy alone.
5. Compare against a simple baseline.
6. Save the entire pipeline so new data goes through identical preprocessing.

**Limitations:** Historical observational data; potentially shifting coding practices; many categorical features; no external validation, calibration, or prospective evaluation. These metrics do not establish clinical safety or usefulness.

**Dataset:** [UCI Diabetes 130-US Hospitals (1999–2008)](https://archive.ics.uci.edu/dataset/296/diabetes+130+us+hospitals+for+years+1999+2008).
